In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Operator backpropagation (OBP) untuk estimasi nilai ekspektasi

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimasi penggunaan: 16 menit pada prosesor Eagle r3 (CATATAN: Ini hanya estimasi. Waktu berjalan kamu bisa berbeda.)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Latar Belakang
Operator backpropagation adalah teknik yang melibatkan penyerapan operasi dari akhir Circuit kuantum ke dalam observable yang diukur, sehingga secara umum mengurangi kedalaman Circuit dengan biaya berupa tambahan suku pada observable. Tujuannya adalah untuk membackpropagasi sebanyak mungkin Circuit tanpa membiarkan observable tumbuh terlalu besar. Implementasi berbasis Qiskit tersedia di addon OBP Qiskit; detail selengkapnya bisa ditemukan di [dokumentasi](/guides/qiskit-addons-obp) terkait beserta [contoh sederhana](/guides/qiskit-addons-obp-get-started) untuk memulai.

Perhatikan contoh Circuit yang di dalamnya observable $O = \sum_P c_P P$ akan diukur, di mana $P$ adalah Pauli dan $c_P$ adalah koefisien. Mari kita notasikan Circuit tersebut sebagai uniter tunggal $U$ yang secara logis dapat dibagi menjadi $U = U_C U_Q$ seperti yang ditunjukkan pada gambar di bawah.

![Diagram Circuit yang menunjukkan Uq diikuti oleh Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

Operator backpropagation menyerap uniter $U_C$ ke dalam observable dengan mengevolusinya sebagai $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$. Dengan kata lain, sebagian komputasi dilakukan secara klasikal melalui evolusi observable dari $O$ ke $O'$. Masalah semula kini dapat diformulasikan ulang sebagai pengukuran observable $O'$ untuk Circuit baru yang memiliki kedalaman lebih rendah dengan uniter $U_Q$.

Uniter $U_C$ direpresentasikan sebagai sejumlah irisan $U_C = U_S U_{S-1}...U_2U_1$. Ada beberapa cara untuk mendefinisikan sebuah irisan. Misalnya, pada Circuit contoh di atas, setiap lapisan $R_{zz}$ dan setiap lapisan Gate $R_x$ dapat dianggap sebagai irisan individual. Backpropagation melibatkan kalkulasi $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$ secara klasikal. Setiap irisan $U_s$ dapat direpresentasikan sebagai $U_s = exp(\frac{-i\theta_s P_s}{2})$, di mana $P_s$ adalah Pauli $n$-Qubit dan $\theta_s$ adalah skalar. Mudah untuk diverifikasi bahwa

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

Pada contoh di atas, jika ${P,P_s} = 0$, maka kita perlu menjalankan dua Circuit kuantum, alih-alih satu, untuk menghitung nilai ekspektasi. Oleh karena itu, backpropagation dapat meningkatkan jumlah suku dalam observable, sehingga menyebabkan jumlah eksekusi Circuit yang lebih tinggi. Salah satu cara untuk memungkinkan backpropagation lebih dalam ke Circuit, sambil mencegah operator tumbuh terlalu besar, adalah dengan memangkas suku-suku yang memiliki koefisien kecil, daripada menambahkannya ke operator. Misalnya, pada contoh di atas, seseorang dapat memilih untuk memangkas suku yang melibatkan $P_sP$ selama $\theta_s$ cukup kecil. Memangkas suku dapat menghasilkan lebih sedikit Circuit kuantum yang perlu dieksekusi, namun hal ini menghasilkan beberapa kesalahan dalam kalkulasi nilai ekspektasi akhir yang sebanding dengan besarnya koefisien suku-suku yang dipangkas.

Tutorial ini mengimplementasikan [pola Qiskit](/guides/intro-to-patterns) untuk mensimulasikan dinamika kuantum rantai spin Heisenberg menggunakan <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>.
## Persyaratan
Sebelum memulai tutorial ini, pastikan kamu telah menginstal hal-hal berikut:

- Qiskit SDK v1.2 atau lebih baru (`pip install qiskit`)
- Qiskit Runtime v0.28 atau lebih baru (`pip install qiskit-ibm-runtime`)
- OBP Qiskit addon (`pip install qiskit-addon-obp`)
- Qiskit addon utils (`pip install qiskit-addon-utils`)
## Pengaturan

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

## Bagian I: Rantai spin Heisenberg skala kecil
### Langkah 1: Petakan input klasikal ke masalah kuantum
#### Petakan evolusi waktu model kuantum Heisenberg ke eksperimen kuantum.
Paket [qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) menyediakan beberapa fungsionalitas yang dapat digunakan kembali untuk berbagai tujuan.

Modul [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators)-nya menyediakan fungsi untuk menghasilkan Hamiltonian seperti Heisenberg pada graf konektivitas tertentu.
Graf ini bisa berupa [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) atau [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap) sehingga mudah digunakan dalam alur kerja berbasis Qiskit.

Berikut ini, kita membuat `CouplingMap` rantai linier dari 10 Qubit.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

Selanjutnya, kita membuat operator Pauli yang memodelkan Hamiltonian Heisenberg XYZ.

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

Di mana $G(V,E)$ adalah graf dari coupling map yang diberikan.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

#### Create circuit slices to backpropagate

The `backpropagate` function backpropagates entire circuit slices at a time. Therefore, the choice of slicing can have an impact on how well backpropagation performs for a given problem. Here, we will group gates of the same type into slices using the [`slice_by_depth`](/docs/api/qiskit-addon-utils/slicing#slice_by_depth) function.

For a more detailed discussion on circuit slicing, check out this [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) of the [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) package.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


Dari operator Qubit, kita bisa membuat Circuit kuantum yang memodelkan evolusi waktunya.
Sekali lagi, modul [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) hadir dengan fungsi yang tepat untuk melakukan hal tersebut:

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### Langkah 2: Optimalkan masalah untuk eksekusi hardware kuantum
#### Buat irisan Circuit untuk dibackpropagasi
Ingat, fungsi ``backpropagate`` akan membackpropagasi seluruh irisan Circuit sekaligus, sehingga pilihan cara memotong dapat berdampak pada seberapa baik backpropagation bekerja untuk masalah tertentu. Di sini, kita akan mengelompokkan Gate dengan tipe yang sama ke dalam irisan menggunakan fungsi [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types).

Untuk diskusi lebih detail tentang pemotongan Circuit, cek [panduan how-to](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) dari paket [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils).

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

Below you will see that we backpropagated six slices, and the terms were combined into six and not eight groups. This implies that backpropagating one more slice would cause the number of Pauli groups to exceed eight. We can verify that this is the case by inspecting the returned metadata. Also note that in this portion the circuit transformation is exact.  That is, no terms of the new observable $O’$ were truncated. The backpropagated circuit and the backpropagated operator give the exact outcome as the original circuit and operator.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### Batasi seberapa besar operator boleh tumbuh selama backpropagation
Selama backpropagation, jumlah suku dalam operator secara umum akan mendekati $4^N$ dengan cepat, di mana $N$ adalah jumlah Qubit. Ketika dua suku dalam operator tidak berkomutasi secara qubit-wise, kita memerlukan Circuit terpisah untuk mendapatkan nilai ekspektasi yang sesuai dengan keduanya. Misalnya, jika kita memiliki observable 2-Qubit $O = 0.1 XX + 0.3 IZ - 0.5 IX$, maka karena $[XX,IX] = 0$, pengukuran dalam satu basis sudah cukup untuk menghitung nilai ekspektasi kedua suku ini. Namun, $IZ$ anti-berkomutasi dengan dua suku lainnya. Jadi kita memerlukan pengukuran basis terpisah untuk menghitung nilai ekspektasi $IZ$. Dengan kata lain, kita memerlukan dua, bukan satu, Circuit untuk menghitung $\langle O \rangle$. Seiring bertambahnya jumlah suku dalam operator, ada kemungkinan jumlah eksekusi Circuit yang diperlukan juga meningkat.

Ukuran operator dapat dibatasi dengan menentukan argumen kata kunci ``operator_budget`` dari fungsi ``backpropagate``, yang menerima instance [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget).

Untuk mengontrol jumlah sumber daya (waktu) tambahan yang dialokasikan, kita membatasi jumlah maksimum grup Pauli yang berkomutasi secara qubit-wise yang boleh dimiliki observable yang telah dibackpropagasi. Di sini kita tentukan bahwa backpropagation harus berhenti ketika jumlah grup Pauli yang berkomutasi secara qubit-wise dalam operator melebihi 8.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

### Step 3: Execute using Qiskit primitives

First, we create two [Primitive Unified Blocs](/docs/api/qiskit/primitives) (PUBs) corresponding to the original circuit, and the backpropagated circuit. Then we execute the pubs on an ideal Estimator to obtain the expectation values.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

### Step 4: Post-process and return result in desired classical format

Now we obtain the expectation values of the original and backpropagated circuits.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Perlu dicatat bahwa dengan mengalokasikan kesalahan `5e-3` per irisan untuk pemangkasan, kita dapat menghapus 1 irisan lagi dari Circuit, sambil tetap berada dalam anggaran asli berupa delapan grup Pauli yang berkomutasi dalam observable. Secara default, `backpropagate` menggunakan norma L1 dari koefisien yang dipangkas untuk membatasi total kesalahan yang ditimbulkan dari pemangkasan. Untuk opsi lain, lihat [panduan how-to tentang menentukan p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html).

Pada contoh khusus ini di mana kita telah membackpropagasi tujuh irisan, total kesalahan pemangkasan seharusnya tidak melebihi ``(5e-3 error/irisan) * (7 irisan) = 3.5e-2``.
Untuk diskusi lebih lanjut tentang mendistribusikan anggaran kesalahan di seluruh irisan, cek [panduan how-to ini](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html).

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution)
- [Multi-product formulas to reduce Trotter error](/docs/tutorials/multi-product-formula)
- [`pauli-prop`](https://github.com/Qiskit/pauli-prop), a Rust-accelerated package for Pauli propagation, with [tutorials](https://github.com/Qiskit/pauli-prop/tree/main/docs/tutorials) covering OBP, classical expectation-value estimation, and noisy simulation
</Admonition>